# 02 Model prototyping
Quick baselines vs LightGBM on a short synthetic slice. Full runs live in `main.py`.

In [1]:
import sys
import tempfile
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
ROOT = ROOT if (ROOT / 'pyproject.toml').exists() else ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import Settings
from src.data_loader import generate_synthetic
from src.evaluate import compute_metrics
from src.feature_engineering import make_features
from src.grid_builder import build_hourly_grid
from src.models import PersistenceBaseline, train_lgbm

cfg = Settings()
tmp = Path(tempfile.gettempdir()) / 'proto_alerts.csv'
df = generate_synthetic(tmp, days=120, seed=7)
grid = build_hourly_grid(df, ['Kyiv', *cfg.neighbors])
X, y = make_features(grid, 'Kyiv', cfg.neighbors)
cut = X.index.max() - pd.Timedelta(days=14)
gap = cut - pd.Timedelta(hours=cfg.horizon_h)
Xtr, Xte = X[X.index <= gap], X[X.index > cut]
ytr, yte = y[y.index <= gap], y[y.index > cut]
ref = float(ytr.mean())
m = train_lgbm(Xtr, ytr, cfg.resolved_lgbm_params())
print('lgbm:', compute_metrics(yte, m.predict_proba(Xte)[:, 1], ref=ref))
print('persist:', compute_metrics(yte, PersistenceBaseline().predict_proba(Xte)[:, 1], ref=ref))


2026-10-11 22:35:08.385 | INFO     | src.data_loader:generate_synthetic:345 - synthetic rows=512 -> C:\Users\User\AppData\Local\Temp\proto_alerts.csv


2026-10-11 22:35:08.430 | INFO     | src.grid_builder:build_hourly_grid:83 - grid: 2883h x 5 regions


lgbm: {'pr_auc': 0.7640531448783766, 'roc_auc': 0.8212230644073177, 'brier': 0.15264031784623336, 'bss': 0.3186282183873511, 'ece': 0.07255765440089645, 'precision': 0.7555555555555555, 'recall': 0.6017699115044248, 'f2': 0.6273062730627307, 'threshold': 0.5}
persist: {'pr_auc': 0.6046660878981367, 'roc_auc': 0.7495535537124489, 'brier': 0.19047619047619047, 'bss': 0.14973250127600668, 'ece': 0.19047619047619047, 'precision': 0.810126582278481, 'recall': 0.5663716814159292, 'f2': 0.6026365348399246, 'threshold': 0.5}
